<a href="https://colab.research.google.com/github/AnitaPannerselvam/cdwmain/blob/day4/Day4_Task_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Import the datasets and transformers packages
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
import numpy as np
import pandas as pd

# Load the sms_spam dataset
# The sms_spam dataset only has a 'train' split.
# We load the 'train' split and then manually split it into train and test.
ds = load_dataset("sms_spam", split="train")

# Split the dataset into train and test
train_test_split = ds.train_test_split(test_size=0.2)  # Adjust test_size as needed
ds = {
    "train": train_test_split["train"],
    "test": train_test_split["test"],
}


# Load a pre-trained BERT model and add a classification head
model_name = "bert-base-uncased"  # Specify the desired pre-trained model name
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,  # Binary classification (spam or not spam)
    id2label={0: "HAM", 1: "SPAM"},
    label2id={"HAM": 0, "SPAM": 1},
)

# Freeze the base model parameters
for param in model.base_model.parameters():
    param.requires_grad = False

# Define the compute metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": (predictions == labels).mean()}

# Before initializing the data collator, you need to initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name) # using the same model name

# Initialize a data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Set training arguments
training_args = TrainingArguments(
    output_dir="./data/sms_spam_classification",
    learning_rate=3e-5,  # Adjust learning rate for experimentation
    per_device_train_batch_size=8,  # Adjust batch size for experimentation
    per_device_eval_batch_size=8,
    num_train_epochs=3,  # Train for 3 epochs
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_steps=10,
)


# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples['sms'], padding="max_length", truncation=True) # changed 'text' to 'sms'

# Applying .map() to the train and test datasets within the dictionary
tokenized_ds = {
    "train": ds["train"].map(tokenize_function, batched=True),
    "test": ds["test"].map(tokenize_function, batched=True),
}

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

# Evaluate the model
results = trainer.evaluate()
print("Evaluation Results:", results)

# Add predictions to the dataset for analysis
predictions = trainer.predict(tokenized_ds["test"])

df = pd.DataFrame(ds["test"])
df["predicted_label"] = np.argmax(predictions[0], axis=1)
df["predicted_label"] = df["predicted_label"].map({0: "HAM", 1: "SPAM"})
df["label"] = df["label"].map({0: "HAM", 1: "SPAM"})

# Display mismatched predictions
mismatched = df[df["label"] != df["predicted_label"]]
pd.set_option("display.max_colwidth", None)
print("Mismatched Predictions:")
print(mismatched.head(5))

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Map:   0%|          | 0/4459 [00:00<?, ? examples/s]

Map:   0%|          | 0/1115 [00:00<?, ? examples/s]

<ipython-input-9-21e90bb532a0>:72: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.311800,0.341878,0.873543
2,0.247000,0.322985,0.873543
3,0.389000,0.316750,0.873543


Evaluation Results: {'eval_loss': 0.31674957275390625, 'eval_accuracy': 0.873542600896861, 'eval_runtime': 31.241, 'eval_samples_per_second': 35.69, 'eval_steps_per_second': 4.481, 'epoch': 3.0}
Mismatched Predictions:
                                                                                                                                                                 sms  \
0              Please call our customer service representative on FREEPHONE 0808 145 4742 between 9am-11pm as you have WON a guaranteed £1000 cash or £5000 prize!\n   
8                          XCLUSIVE@CLUBSAISAI 2MOROW 28/5 SOIREE SPECIALE ZOUK WITH NICHOLS FROM PARIS.FREE ROSES 2 ALL LADIES !!! info: 07946746291/07880867867 \n   
10  UR GOING 2 BAHAMAS! CallFREEFONE 08081560665 and speak to a live operator to claim either Bahamas cruise of£2000 CASH 18+only. To opt out txt X to 07786200117\n   
11                    Free 1st week entry 2 TEXTPOD 4 a chance 2 win 40GB iPod or £250 cash every wk. Txt POD